In [1]:
# 07_shap_vs_dice.ipynb
# Structural comparison of SHAP (feature attribution) and DiCE (counterfactual recourse)
# on a matched set of high-risk non-diabetic cases (20 male + 20 female). Computes TRUE
# SHAP values with TreeExplainer (resolving the earlier gain-importance mislabelling),
# the DiCE changed-feature set per case, their Jaccard overlap, and whether each method
# supplies concrete numeric target values. Produces a grayscale SHAP summary figure and a
# comparison table. dpi 600, PNG+PDF, no titles/captions.

import warnings, signal
from pathlib import Path
import numpy as np
import pandas as pd
import xgboost as xgb
import shap
import dice_ml
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
class _Timeout(Exception):
    pass
def _handler(signum, frame):
    raise _Timeout()
signal.signal(signal.SIGALRM, _handler)
PER_CASE_TIMEOUT = 6
SEED = 42; np.random.seed(SEED)
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"; TAB = ROOT / "results" / "tables"; FIG = ROOT / "results" / "figures"; MODELS = ROOT / "models"

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"savefig.dpi": 600, "figure.dpi": 120, "pdf.fonttype": 42,
                     "ps.fonttype": 42, "font.size": 11, "axes.edgecolor": "0.2",
                     "axes.linewidth": 0.8, "grid.color": "0.85"})
def savefig(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight"); plt.close(fig)

PRED = ["age", "HE_BMI", "HE_wc", "HE_wt", "HE_ht", "N_EN", "N_CHO", "N_SUGAR",
        "N_CHOL", "sm_presnt", "dr_month", "pa_aerobic", "L_BR_FQ", "edu", "ho_incm", "D_1_1"]
CATEG = ["sm_presnt", "dr_month", "pa_aerobic"]; CONT = [c for c in PRED if c not in CATEG]
IMMUTABLE = ["age", "HE_ht", "edu", "ho_incm", "D_1_1"]; VARY = [c for c in PRED if c not in IMMUTABLE]
WAIST_CAP = 15.0; PCT_CAP = {"HE_BMI": 0.20, "HE_wt": 0.20}; N_PER_SEX = 20; TOPK = 3

def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / len(a | b) if (a | b) else 0.0

rows = []; shap_abs_all = {}; eval_rows = []
for sx in ["Male", "Female"]:
    full = pd.read_parquet(DATA / f"scored_{sx.lower()}.parquet")
    deploy = xgb.XGBClassifier(); deploy.load_model(MODELS / f"xgb_{sx.lower()}_deploy.json")
    expl = shap.TreeExplainer(deploy)
    hr_nd = pd.read_parquet(DATA / f"highrisk_nondiab_{sx.lower()}.parquet")
    samp = hr_nd.sample(min(N_PER_SEX, len(hr_nd)), random_state=SEED).reset_index(drop=True)
    sv = expl.shap_values(samp[PRED].astype(float))
    shap_abs_all[sx] = np.abs(sv).mean(axis=0)
    # DiCE setup
    data_df = full[PRED + ["diabetes"]].copy()
    for c in CATEG:
        data_df[c] = data_df[c].astype(int).astype(str)
    class Wrap:
        def __init__(self, mm): self.m = mm
        def predict_proba(self, X):
            Z = X.copy()
            for c in CATEG:
                if Z[c].dtype == object:
                    Z[c] = Z[c].astype(float)
            return self.m.predict_proba(Z[PRED].astype(float))
        def predict(self, X):
            return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)
    d = dice_ml.Data(dataframe=data_df, continuous_features=CONT, outcome_name="diabetes")
    m = dice_ml.Model(model=Wrap(deploy), backend="sklearn", model_type="classifier")
    exp = dice_ml.Dice(d, m, method="random")
    for i in range(len(samp)):
        x = samp.iloc[i]
        shap_top = [PRED[t] for t in np.argsort(-np.abs(sv[i]))[:TOPK]]
        row = samp.iloc[[i]][PRED].copy()
        for c in CATEG:
            row[c] = row[c].astype(int).astype(str)
        pr = {"HE_wc": [float(x.HE_wc - WAIST_CAP), float(x.HE_wc)]}
        for v, cap in PCT_CAP.items():
            pr[v] = [float(x[v] * (1 - cap)), float(x[v])]
        dice_changed = []; best = None; op = float(deploy.predict_proba(samp.iloc[[i]][PRED].astype(float))[:, 1][0]); cp = op
        try:
            signal.alarm(PER_CASE_TIMEOUT)
            cf = exp.generate_counterfactuals(row, total_CFs=4, desired_class=0,
                                              features_to_vary=VARY, permitted_range=pr, random_seed=SEED)
            signal.alarm(0)
            cfdf = cf.cf_examples_list[0].final_cfs_df
            if cfdf is not None and len(cfdf):
                Xcf = cfdf[PRED].astype(float)
                probs = deploy.predict_proba(Xcf)[:, 1]
                best = Xcf.iloc[int(np.argmin(probs))]; cp = float(np.min(probs))
                dice_changed = [v for v in VARY if abs(float(best[v]) - float(x[v])) > 1e-6]
        except (_Timeout, Exception):
            signal.alarm(0)
        rows.append([sx, i, "|".join(shap_top), "|".join(dice_changed[:TOPK]),
                     round(jaccard(shap_top, dice_changed), 3),
                     "age" in shap_top, len(dice_changed) > 0])
        # ---- export row for the LLM guide-generation / evaluation notebooks ----
        er = {"sex": sx, "case_id": f"{sx[:1]}{i:02d}", "orig_prob": round(op, 4), "dice_cf_prob": round(cp, 4),
              "shap_top3": "|".join(shap_top), "dice_changed": "|".join(dice_changed)}
        for v in PRED:
            er[f"orig_{v}"] = round(float(x[v]), 2)
            er[f"cf_{v}"] = round(float(best[v]), 2) if best is not None else round(float(x[v]), 2)
        eval_rows.append(er)

cmp = pd.DataFrame(rows, columns=["Sex", "case", "SHAP_top3", "DiCE_changed_top3",
                                  "Jaccard_overlap", "SHAP_includes_age", "DiCE_has_target"])
cmp.to_csv(TAB / "table_shap_dice_case_level.csv", index=False)

summary = pd.DataFrame({
    "Method": ["SHAP (attribution)", "DiCE (counterfactual)"],
    "Provides_numeric_target": ["No", "Yes"],
    "Mean_feature_overlap_Jaccard": [round(cmp.Jaccard_overlap.mean(), 3), "ref"],
    "Share_top_feature_immutable_age": [round(cmp.SHAP_includes_age.mean(), 3), 0.0],
    "Cases_with_actionable_target": [0, int(cmp.DiCE_has_target.sum())],
})
summary.to_csv(TAB / "table_shap_dice_comparison.csv", index=False)
print("SHAP vs DiCE comparison:")
print(summary.to_string(index=False))
print(f"\nMean SHAP-DiCE feature-overlap Jaccard = {cmp.Jaccard_overlap.mean():.3f} "
      f"(0 => methods identify different feature sets)")

# ---- Grayscale SHAP mean-|value| summary (both sexes) ----
imp = pd.DataFrame({"feature": PRED, "Male": shap_abs_all["Male"], "Female": shap_abs_all["Female"]})
imp = imp.sort_values("Male", ascending=True)
fig, ax = plt.subplots(figsize=(7, 5))
yy = np.arange(len(imp))
ax.barh(yy - 0.2, imp["Male"], height=0.4, color="#3b3b3b", edgecolor="black", linewidth=0.5, label="Male")
ax.barh(yy + 0.2, imp["Female"], height=0.4, color="#a9a9a9", edgecolor="black", linewidth=0.5, hatch="///", label="Female")
ax.set_yticks(yy); ax.set_yticklabels(imp["feature"]); ax.set_xlabel("Mean |SHAP value|")
ax.legend(frameon=False)
savefig(fig, "fig_shap_importance")

# ---- Export the 40 evaluation cases (20 male + 20 female) for the API notebooks ----
ev = pd.DataFrame(eval_rows)
ev.to_parquet(DATA / "eval_cases.parquet", index=False)
ev.to_csv(TAB / "eval_cases.csv", index=False)
print(f"Saved SHAP vs DiCE tables, SHAP importance figure, and {len(ev)} evaluation cases.")


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:05<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  2.26it/s]

100%|██████████| 1/1 [00:00<00:00,  2.26it/s]

...

100%|██████████| 1/1 [00:00<00:00,  2.05it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:05<?, ?it/s]

SHAP vs DiCE comparison:
               Method Provides_numeric_target Mean_feature_overlap_Jaccard  Share_top_feature_immutable_age  Cases_with_actionable_target
   SHAP (attribution)                      No                        0.115                              1.0                             0
DiCE (counterfactual)                     Yes                          ref                              0.0                            32

Mean SHAP-DiCE feature-overlap Jaccard = 0.115 (0 => methods identify different feature sets)


Saved SHAP vs DiCE tables, SHAP importance figure, and 40 evaluation cases.
